In [13]:
#상권분석 사이트 자동 실행
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
driver.get("https://golmok.seoul.go.kr/intendedOwner/intendedOwner.do")

In [14]:
# 현재 열려 있는 창 목록 확인 후, 가장 최근에 뜬 '리포트 창'으로 제어권 이동
driver.switch_to.window(driver.window_handles[-1])
print("이제 분석리포트 창을 조종합니다!")

이제 분석리포트 창을 조종합니다!


In [18]:
try:
    # 요약 정보들이 들어있는 영역 찾기
    summary_list = driver.find_elements(By.CSS_SELECTOR, ".report_summary li")
    
    print("--- 분석 요약 결과 ---")
    for item in summary_list:
        print(item.text) # "둔촌1동의 커피-음료 점포수가..." 등의 문구가 출력됩니다.
        
except Exception as e:
    print(f"요약 데이터를 가져오는데 실패했어요: {e}")

--- 분석 요약 결과 ---


In [17]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

try:
    print("--- 분석 요약 문구 수집 재시도 ---")
    
    # [개선 1] presence(존재) 대신 visibility(화면 표시)를 기다립니다.
    # 사용자 눈에 보일 때까지 기다려야 .text 추출이 더 안정적입니다.
    WebDriverWait(driver, 15).until(
        EC.visibility_of_element_located((By.ID, "summary01"))
    )
    
    # (선택 사항) 다른 요약들이 로딩될 시간을 아주 잠깐 줍니다. (동적 페이지인 경우)
    # time.sleep(1) 
    
    # [개선 2] XPath를 조금 더 구체적으로 수정하는 것을 권장합니다.
    # 만약 요약 문구가 모두 <p> 태그나 <span> 태그라면 '*' 대신 태그명을 쓰세요.
    # 예: "//p[contains(@id, 'summary')]" 또는 "//div[starts-with(@id, 'summary')]"
    # 여기서는 ID가 'summary'로 '시작하는' 요소로 한정해 봅니다.
    summaries = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'summary')]")
    
    print(f"찾은 요소 개수: {len(summaries)}개")
    
    for s in summaries:
        # [개선 3] .text가 비어있을 경우를 대비해 textContent(숨겨진 텍스트)도 확인
        content = s.text.strip()
        if not content:
            content = s.get_attribute("textContent").strip()
            
        if content:
            # ID도 같이 출력해서 어떤 녀석이 잡혔는지 확인하면 디버깅에 좋습니다.
            element_id = s.get_attribute("id")
            print(f"✅ [{element_id}] 추출 데이터: {content}")
        else:
            print(f"⚠️ 요소는 찾았으나 텍스트가 비어있음 (ID: {s.get_attribute('id')})")
            
except Exception as e:
    print(f"❌ 오류 발생: {e}")


--- 분석 요약 문구 수집 재시도 ---
찾은 요소 개수: 4개
✅ [summary01] 추출 데이터: 커피-음료의 점포수가 전분기대비 증가하고 있습니다. 상권이 발달하는 시기인 경우 입지선정에 신중하셔야 합니다.
✅ [summary02] 추출 데이터: 둔촌1동은 자치구에 비해 매출이 증가 추세입니다. 인근지역에 비해 활성화된 상권입니다. 경쟁관계의 유의하세요.매출액은 카드사용액을 기반으로 추정된 값으로 평균으로만 제공합니다.실제점포의 매출과 차이가 있을 수 있으므로 참고자료로 활용하세요.
✅ [summary03] 추출 데이터: 둔촌1동은 전년 동분기에 비해 유동인구가 증가하고있는 지역입니다. 경쟁 업소출현을 경계하세요.
✅ [summary04] 추출 데이터: 자치구 내 행정동 18개 중 둔촌1동의 점포수는 18위, 매출액 3위, 유동인구 18위 입니다.


In [35]:
# 점포수 가져오기
target_box = driver.find_element(By.ID, "reportMain")
print(target_box.text)

전분기 대비
725만원
2025년 3분기
1,159만원
전년 동분기 대비
542만원
나의 등수 3/18위
유동인구 (3개월간 인구수)
전분기 대비
14명
2025년 3분기
329명/ha
전년 동분기 대비
240명
나의 등수 18/18위


In [46]:
import time
from selenium.webdriver.common.by import By

# 1. 조종권을 '리포트 팝업창'으로 확실히 옮깁니다.
driver.switch_to.window(driver.window_handles[-1])
time.sleep(2)

# 2. [핵심] 모든 iframe(방)을 무시하고, '내용'이 있는 방으로 강제 진입합니다.
try:
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    if len(iframes) > 0:
        driver.switch_to.frame(iframes[0]) # 첫 번째 iframe 방으로 입장
        print("✅ 리포트 속창 진입 성공!")
    
    # 3. [수정] reportDetail 대신 사진에서 확인된 reportMain으로 찾기
    # 승둥이님의 사진 image_03ab81.png에 적힌 ID입니다.
    target_data = driver.find_element(By.ID, "reportMain").text
    
    print("--- 📋 긁어온 데이터 결과 ---")
    print(target_data)
    
except Exception as e:
    print(f"❌ 여전히 못 찾고 있어요. 이유: {e}")

❌ 여전히 못 찾고 있어요. 이유: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="reportMain"]"}
  (Session info: chrome=144.0.7559.110); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff751aca535
	0x7ff751aca590
	0x7ff7518710bd
	0x7ff7518cc48e
	0x7ff7518cc79c
	0x7ff75191d3a7
	0x7ff751919ef5
	0x7ff7518bcc9c
	0x7ff7518bdbe3
	0x7ff751da7b60
	0x7ff751da1f5d
	0x7ff751dc311a
	0x7ff751ae60a5
	0x7ff751aee73c
	0x7ff751ad3844
	0x7ff751ad39f6
	0x7ff751ab9a07
	0x7ffd73dfe8d7
	0x7ffd751ac53c

